# CUAD RAG Chatbot — Person A: Data Ingestion Pipeline

This notebook implements all six ingestion steps:
- **A-1** — Explore the JSON structure
- **A-2** — Extract chunks with metadata
- **A-3** — Embed chunks with `BAAI/bge-large-en-v1.5`
- **A-4** — Load vectors into Qdrant
- **A-5** — Build BM25 keyword index
- **A-6** — Export metadata CSV for evaluation

**Prerequisites — run this once before the notebook:**
```bash
pip install sentence-transformers qdrant-client rank-bm25
docker run -d -p 6333:6333 qdrant/qdrant
```

## 0 · Imports & paths

In [1]:
pip install sentence-transformers qdrant-client rank-bm25

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json, re, csv, pickle, os
from pathlib import Path

# ── paths ──────────────────────────────────────────────────────────────────
ROOT          = Path(".")                          # repo root (run from here)
DATA_DIR      = ROOT / "CUAD_v1"
JSON_PATH     = DATA_DIR / "CUAD_v1.json"
CACHE_PATH    = ROOT / "ingestion" / "embeddings_cache.pkl"
BM25_PATH     = ROOT / "ingestion" / "bm25_index.pkl"
META_CSV_PATH = ROOT / "evaluation" / "chunks_metadata.csv"

# create output directories if they don't exist
for d in [ROOT / "ingestion", ROOT / "evaluation"]:
    d.mkdir(parents=True, exist_ok=True)

print("JSON exists:", JSON_PATH.exists())
print("Output dirs ready: ingestion/ evaluation/")

JSON exists: True
Output dirs ready: ingestion/ evaluation/


---
## Step A-1 — Explore the JSON structure

In [3]:
with open(JSON_PATH, encoding="utf-8") as f:
    raw_data = json.load(f)

print(f"Top-level keys : {list(raw_data.keys())}")
print(f"Number of contracts : {len(raw_data['data'])}")

Top-level keys : ['version', 'data']
Number of contracts : 510


In [4]:
# Inspect the first contract
contract = raw_data["data"][0]
print(f"Contract title      : {contract['title']}")
print(f"Number of paragraphs: {len(contract['paragraphs'])}")

para = contract["paragraphs"][0]
print(f"\nParagraph text (first 300 chars):\n{para['context'][:300]}")

Contract title      : LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT
Number of paragraphs: 1

Paragraph text (first 300 chars):
EXHIBIT 10.6

                              DISTRIBUTOR AGREEMENT

         THIS  DISTRIBUTOR  AGREEMENT (the  "Agreement")  is made by and between Electric City Corp.,  a Delaware  corporation  ("Company")  and Electric City of Illinois LLC ("Distributor") this 7th day of September, 1999.

        


In [5]:
# Inspect a QA pair
qa = para["qas"][0]
print(f"Clause category : {qa['id']}")
print(f"Question        : {qa['question']}")
print(f"Clause present  : {not qa['is_impossible']}")
print(f"Answer spans    : {qa['answers']}")
print(f"\nTotal QA pairs per paragraph: {len(para['qas'])} (41 clause categories)")

Clause category : LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT__Document Name
Question        : Highlight the parts (if any) of this contract related to "Document Name" that should be reviewed by a lawyer. Details: The name of the contract
Clause present  : True
Answer spans    : [{'text': 'DISTRIBUTOR AGREEMENT', 'answer_start': 44}]

Total QA pairs per paragraph: 41 (41 clause categories)


In [6]:
# Print all 41 clause category names
categories = [qa["id"] for qa in para["qas"]]
for i, cat in enumerate(categories, 1):
    print(f"  {i:2d}. {cat}")

   1. LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT__Document Name
   2. LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT__Parties
   3. LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT__Agreement Date
   4. LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT__Effective Date
   5. LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT__Expiration Date
   6. LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT__Renewal Term
   7. LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT__Notice Period To Terminate Renewal
   8. LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT__Governing Law
   9. LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT__Most Favored Nation
  10. LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT__Non-Compete
  11. LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT__Exclusivity
  12. LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT__No-Solicit Of Customers
  13. LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT__Competitive Restriction Exception
 

In [7]:
# Quick dataset statistics
total_paragraphs = sum(len(c["paragraphs"]) for c in raw_data["data"])
total_positives  = sum(
    1
    for c in raw_data["data"]
    for p in c["paragraphs"]
    for qa in p["qas"]
    if not qa["is_impossible"] and qa["answers"]
)
print(f"Total contracts  : {len(raw_data['data'])}")
print(f"Total paragraphs : {total_paragraphs:,}")
print(f"Total positive QA spans : {total_positives:,}")

Total contracts  : 510
Total paragraphs : 510
Total positive QA spans : 6,702


---
## Step A-2 — Extract chunks with metadata

In [8]:
def clean_text(text: str) -> str:
    """Remove CUAD-specific artifacts from contract text."""
    text = text.replace("<omitted>", " ... ")      # annotators used this for skipped text
    text = re.sub(r'\[?\*+\]?', '[REDACTED]', text) # redacted sections appear as ***
    text = re.sub(r'_{3,}',      '[REDACTED]', text) # also appear as ___
    text = re.sub(r'\s+',        ' ',          text) # normalize whitespace from PDF conversion
    return text.strip()


def extract_chunks(data: dict, min_len: int = 50) -> list[dict]:
    """Convert every paragraph into a chunk dict with clause metadata."""
    chunks = []
    skipped = 0

    for contract in data["data"]:
        contract_name = contract["title"]

        for chunk_index, para in enumerate(contract["paragraphs"]):
            text = clean_text(para["context"])

            # skip very short fragments — usually headers or page numbers
            if len(text) < min_len:
                skipped += 1
                continue

            # collect clause categories present in this paragraph
            # is_impossible=False  → clause found here
            # is_impossible=True   → clause NOT here
            clause_categories = []
            answers           = []
            for qa in para["qas"]:
                if not qa["is_impossible"] and qa["answers"]:
                    clause_categories.append(qa["id"])
                    answers.append(qa["answers"][0]["text"])

            chunks.append({
                "contract_name":     contract_name,
                "chunk_index":       chunk_index,
                "text":              text,
                "clause_categories": clause_categories,
                "answers":           answers,
            })

    print(f"Extracted : {len(chunks):,} chunks")
    print(f"Skipped   : {skipped:,} fragments (< {min_len} chars)")
    return chunks

In [9]:
chunks = extract_chunks(raw_data)

# preview a chunk that has clause labels
labeled = [c for c in chunks if c["clause_categories"]]
example = labeled[0]
print(f"Contract      : {example['contract_name']}")
print(f"Chunk index   : {example['chunk_index']}")
print(f"Clause types  : {example['clause_categories']}")
print(f"Answers       : {example['answers']}")
print(f"Text preview  : {example['text'][:300]}")

Extracted : 510 chunks
Skipped   : 0 fragments (< 50 chars)
Contract      : LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT
Chunk index   : 0
Clause types  : ['LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT__Document Name', 'LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT__Parties', 'LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT__Agreement Date', 'LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT__Effective Date', 'LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT__Expiration Date', 'LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT__Renewal Term', 'LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT__Governing Law', 'LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT__Exclusivity', 'LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT__No-Solicit Of Customers', 'LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT__No-Solicit Of Employees', 'LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT__Rofr/Rofo/Rofn', 'LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMEN

In [10]:
# Distribution: how many chunks have ≥1 clause label?
n_labeled   = sum(1 for c in chunks if c["clause_categories"])
n_unlabeled = len(chunks) - n_labeled
print(f"Chunks with ≥1 clause label : {n_labeled:,}")
print(f"Chunks with no label        : {n_unlabeled:,}")

# Most common clause categories across the corpus
from collections import Counter
cat_counter = Counter(
    cat
    for c in chunks
    for cat in c["clause_categories"]
)
print("\nTop 10 clause categories:")
for cat, count in cat_counter.most_common(10):
    print(f"  {count:5,}  {cat}")

Chunks with ≥1 clause label : 510
Chunks with no label        : 0

Top 10 clause categories:
      1  LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT__Document Name
      1  LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT__Parties
      1  LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT__Agreement Date
      1  LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT__Effective Date
      1  LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT__Expiration Date
      1  LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT__Renewal Term
      1  LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT__Governing Law
      1  LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT__Exclusivity
      1  LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT__No-Solicit Of Customers
      1  LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT__No-Solicit Of Employees


---
## Step A-3 — Embed chunks with BAAI/bge-large-en-v1.5

> **Skip if `embeddings_cache.pkl` already exists** — embedding 50k+ chunks takes ~10–30 min on CPU.

In [11]:
if CACHE_PATH.exists():
    print(f"Cache found at {CACHE_PATH} — loading instead of re-embedding.")
    with open(CACHE_PATH, "rb") as f:
        chunks = pickle.load(f)
    print(f"Loaded {len(chunks):,} chunks from cache.")
else:
    print("No cache found — will embed in the next cell.")

Cache found at ingestion\embeddings_cache.pkl — loading instead of re-embedding.
Loaded 510 chunks from cache.


In [12]:
# Run only if cache was NOT found above
if not CACHE_PATH.exists():
    from sentence_transformers import SentenceTransformer

    print("Loading BAAI/bge-large-en-v1.5 …")
    model = SentenceTransformer("BAAI/bge-large-en-v1.5")

    texts = [c["text"] for c in chunks]

    print(f"Embedding {len(texts):,} chunks (batch_size=32) …")
    embeddings = model.encode(
        texts,
        batch_size=32,
        show_progress_bar=True,
        normalize_embeddings=True,   # required for cosine similarity in Qdrant
    )

    for i, chunk in enumerate(chunks):
        chunk["embedding"] = embeddings[i].tolist()

    with open(CACHE_PATH, "wb") as f:
        pickle.dump(chunks, f)

    print(f"\nEmbedding dimension : {len(chunks[0]['embedding'])}")
    print(f"Saved cache to      : {CACHE_PATH}")

In [13]:
# Sanity check
sample = chunks[0]
assert "embedding" in sample, "Embeddings missing — run the cell above first."
print(f"Embedding dimension : {len(sample['embedding'])}")
print(f"First 5 values      : {sample['embedding'][:5]}")

Embedding dimension : 1024
First 5 values      : [0.018912043422460556, 0.03611091524362564, -0.023675385862588882, 0.014576994813978672, -0.03717697784304619]


---
## Step A-4 — Load into Qdrant (embedded local mode)

> **No Docker needed.** `QdrantClient(path=...)` uses Qdrant's built-in embedded engine and persists data on disk inside `qdrant_storage/`. As of `qdrant-client >= 1.7` no extras are required.
>
> To switch to a remote server later, swap the client line to:
> ```python
> client = QdrantClient("localhost", port=6333)
> ```

In [14]:
from pathlib import Path
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointStruct

COLLECTION_NAME  = "cuad_contracts"
VECTOR_DIM       = 1024
BATCH_SIZE       = 256
QDRANT_LOCAL_DIR = str(Path(".") / "qdrant_storage")

# Release any lock held by a previous run of this cell
try:
    client.close()
except Exception:
    pass

client = QdrantClient(path=QDRANT_LOCAL_DIR)
print("Qdrant local client ready.")
print("Existing collections:", [c.name for c in client.get_collections().collections])

Qdrant local client ready.
Existing collections: ['cuad_contracts']


In [15]:
# (Re)create the collection — safe to re-run
if client.collection_exists(COLLECTION_NAME):
    client.delete_collection(COLLECTION_NAME)

client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(
        size=VECTOR_DIM,
        distance=Distance.COSINE,
    ),
)
print(f"Collection '{COLLECTION_NAME}' created.")

Collection 'cuad_contracts' created.


In [16]:
# Build PointStruct list
points = [
    PointStruct(
        id=idx,
        vector=chunk["embedding"],
        payload={
            "contract_name":     chunk["contract_name"],
            "chunk_index":       chunk["chunk_index"],
            "text":              chunk["text"],
            "clause_categories": chunk["clause_categories"],
            "answers":           chunk["answers"],
        },
    )
    for idx, chunk in enumerate(chunks)
]

print(f"Prepared {len(points):,} points for upload.")

Prepared 510 points for upload.


In [17]:
# Upload in batches
for start in range(0, len(points), BATCH_SIZE):
    batch = points[start : start + BATCH_SIZE]
    client.upsert(collection_name=COLLECTION_NAME, points=batch)
    end = min(start + BATCH_SIZE, len(points))
    print(f"  Uploaded {end:,} / {len(points):,}", end="\r")

print()
vec_count = client.get_collection(COLLECTION_NAME).points_count
print(f"Collection '{COLLECTION_NAME}' — {vec_count:,} vectors stored.")

  Uploaded 510 / 510
Collection 'cuad_contracts' — 510 vectors stored.


In [18]:
from sentence_transformers import SentenceTransformer

if "model" not in dir():
    model = SentenceTransformer("BAAI/bge-large-en-v1.5")

test_query = "What is the governing law of this contract?"
q_vec = model.encode(test_query, normalize_embeddings=True).tolist()

results = client.query_points(
    collection_name=COLLECTION_NAME,
    query=q_vec,
    limit=3,
    with_payload=True,
).points

print(f"Query: '{test_query}'\n")
for i, hit in enumerate(results, 1):
    print(f"--- Result {i}  (score={hit.score:.4f}) ---")
    print(f"Contract : {hit.payload['contract_name']}")
    print(f"Clauses  : {hit.payload['clause_categories']}")
    print(f"Answers  : {hit.payload['answers']}")
    print(f"Text     : {hit.payload['text'][:250]}\n")



Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Query: 'What is the governing law of this contract?'

--- Result 1  (score=0.6631) ---
Contract : UAGHINC_04_14_2004-EX-10.18-MAINTENANCE AGREEMENT
Clauses  : ['UAGHINC_04_14_2004-EX-10.18-MAINTENANCE AGREEMENT__Document Name', 'UAGHINC_04_14_2004-EX-10.18-MAINTENANCE AGREEMENT__Parties', 'UAGHINC_04_14_2004-EX-10.18-MAINTENANCE AGREEMENT__Agreement Date', 'UAGHINC_04_14_2004-EX-10.18-MAINTENANCE AGREEMENT__Effective Date', 'UAGHINC_04_14_2004-EX-10.18-MAINTENANCE AGREEMENT__Expiration Date', 'UAGHINC_04_14_2004-EX-10.18-MAINTENANCE AGREEMENT__Renewal Term', 'UAGHINC_04_14_2004-EX-10.18-MAINTENANCE AGREEMENT__Notice Period To Terminate Renewal', 'UAGHINC_04_14_2004-EX-10.18-MAINTENANCE AGREEMENT__Governing Law', 'UAGHINC_04_14_2004-EX-10.18-MAINTENANCE AGREEMENT__Termination For Convenience', 'UAGHINC_04_14_2004-EX-10.18-MAINTENANCE AGREEMENT__Anti-Assignment', 'UAGHINC_04_14_2004-EX-10.18-MAINTENANCE AGREEMENT__Cap On Liability', 'UAGHINC_04_14_2004-EX-10.18-MAINTENANCE AGREEMENT__Ins

---
## Step A-5 — Build BM25 keyword index

In [19]:
from rank_bm25 import BM25Okapi

def tokenize(text: str) -> list[str]:
    return re.findall(r'\b\w+\b', text.lower())


print("Tokenizing corpus …")
corpus_texts  = [c["text"] for c in chunks]
tokenized     = [tokenize(t) for t in corpus_texts]

print("Fitting BM25Okapi …")
bm25 = BM25Okapi(tokenized)

bm25_payload = {
    "bm25":         bm25,
    "corpus_texts": corpus_texts,
    "chunk_ids":    list(range(len(chunks))),
}

with open(BM25_PATH, "wb") as f:
    pickle.dump(bm25_payload, f)

print(f"BM25 index saved to {BM25_PATH}  ({len(chunks):,} documents)")

Tokenizing corpus …
Fitting BM25Okapi …
BM25 index saved to ingestion\bm25_index.pkl  (510 documents)


In [20]:
# BM25 smoke-test
test_query_bm25 = "governing law jurisdiction"
scores     = bm25.get_scores(tokenize(test_query_bm25))
top_ids    = scores.argsort()[::-1][:3]

print(f"BM25 query: '{test_query_bm25}'\n")
for rank, idx in enumerate(top_ids, 1):
    print(f"--- Rank {rank}  (BM25 score={scores[idx]:.4f}) ---")
    print(f"Contract : {chunks[idx]['contract_name']}")
    print(f"Clauses  : {chunks[idx]['clause_categories']}")
    print(f"Text     : {chunks[idx]['text'][:250]}\n")

BM25 query: 'governing law jurisdiction'

--- Rank 1  (BM25 score=7.8661) ---
Contract : Principal Life Insurance Company - Broker Dealer Marketing and Servicing Agreement
Clauses  : ['Principal Life Insurance Company - Broker Dealer Marketing and Servicing Agreement__Document Name', 'Principal Life Insurance Company - Broker Dealer Marketing and Servicing Agreement__Parties', 'Principal Life Insurance Company - Broker Dealer Marketing and Servicing Agreement__Agreement Date', 'Principal Life Insurance Company - Broker Dealer Marketing and Servicing Agreement__Effective Date', 'Principal Life Insurance Company - Broker Dealer Marketing and Servicing Agreement__Governing Law', 'Principal Life Insurance Company - Broker Dealer Marketing and Servicing Agreement__Termination For Convenience', 'Principal Life Insurance Company - Broker Dealer Marketing and Servicing Agreement__Anti-Assignment', 'Principal Life Insurance Company - Broker Dealer Marketing and Servicing Agreement__Insurance']


---
## Step A-6 — Export metadata CSV for Person C

In [21]:
FIELDNAMES = [
    "chunk_id", "contract_name", "chunk_index",
    "clause_category", "answer", "text_preview",
]

rows_written = 0
with open(META_CSV_PATH, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=FIELDNAMES)
    writer.writeheader()

    for idx, chunk in enumerate(chunks):
        base = {
            "chunk_id":      idx,
            "contract_name": chunk["contract_name"],
            "chunk_index":   chunk["chunk_index"],
            "text_preview":  chunk["text"][:150],
        }

        if not chunk["clause_categories"]:
            # paragraph with no labeled clauses — one row with empty category
            writer.writerow({**base, "clause_category": "", "answer": ""})
            rows_written += 1
        else:
            # one row per clause category in this chunk
            for cat, ans in zip(chunk["clause_categories"], chunk["answers"]):
                writer.writerow({**base, "clause_category": cat, "answer": ans})
                rows_written += 1

print(f"Metadata CSV written to : {META_CSV_PATH}")
print(f"Total rows (incl header): {rows_written + 1:,}")

Metadata CSV written to : evaluation\chunks_metadata.csv
Total rows (incl header): 6,703


In [22]:
# Preview the CSV
import pandas as pd

meta_df = pd.read_csv(META_CSV_PATH)
print(f"Shape: {meta_df.shape}")
meta_df.head(10)

Shape: (6702, 6)


,chunk_id,contract_name,chunk_index,clause_category,answer,text_preview
0,0,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,0,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,DISTRIBUTOR AGREEMENT,EXHIBIT 10.6 DISTRIBUTOR AGREEMENT THIS DISTRI...
1,0,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,0,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,Distributor,EXHIBIT 10.6 DISTRIBUTOR AGREEMENT THIS DISTRI...
2,0,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,0,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,"7th day of September, 1999.",EXHIBIT 10.6 DISTRIBUTOR AGREEMENT THIS DISTRI...
3,0,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,0,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,The term of this Agreement shall be ten (10)...,EXHIBIT 10.6 DISTRIBUTOR AGREEMENT THIS DISTRI...
4,0,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,0,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,The term of this Agreement shall be ten (10)...,EXHIBIT 10.6 DISTRIBUTOR AGREEMENT THIS DISTRI...
5,0,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,0,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,If Distributor comp...,EXHIBIT 10.6 DISTRIBUTOR AGREEMENT THIS DISTRI...
6,0,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,0,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,This Agreement is to be construed according to...,EXHIBIT 10.6 DISTRIBUTOR AGREEMENT THIS DISTRI...
7,0,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,0,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,The Distributor shall not order or ...,EXHIBIT 10.6 DISTRIBUTOR AGREEMENT THIS DISTRI...
8,0,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,0,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,Distributor further agrees that it will not in...,EXHIBIT 10.6 DISTRIBUTOR AGREEMENT THIS DISTRI...
9,0,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,0,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,During the Term of this Agreement and for a pe...,EXHIBIT 10.6 DISTRIBUTOR AGREEMENT THIS DISTRI...


In [23]:
# Clause category distribution (for labeled rows only)
labeled_df = meta_df[meta_df["clause_category"] != ""]
print(f"Labeled rows  : {len(labeled_df):,}")
print(f"Unlabeled rows: {(meta_df['clause_category'] == '').sum():,}")
print("\nRows per clause category:")
print(labeled_df["clause_category"].value_counts().to_string())

Labeled rows  : 6,702
Unlabeled rows: 0

Rows per clause category:
clause_category
LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT__Document Name                                                                                                                                  1
LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT__Parties                                                                                                                                        1
LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT__Agreement Date                                                                                                                                 1
LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT__Effective Date                                                                                                                                 1
LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT__Expiration Date                                                         

---
## Handoff summary

| Artifact | Location | Goes to |
|---|---|---|
| Qdrant collection | `localhost:6333` · collection = `cuad_contracts` | Person B |
| BM25 index | `ingestion/bm25_index.pkl` | Person B |
| Metadata CSV | `evaluation/chunks_metadata.csv` | Person C |
| Embedding cache | `ingestion/embeddings_cache.pkl` | (local, skip re-embedding) |
| Vector dimension | **1024**, distance = **COSINE** | Person B |

In [24]:
# Final checklist
checks = {
    "Qdrant collection": client.get_collection(COLLECTION_NAME).vectors_count > 0,
    "BM25 index file":   BM25_PATH.exists(),
    "Metadata CSV":      META_CSV_PATH.exists(),
    "Embeddings cache":  CACHE_PATH.exists(),
}
print("Handoff checklist:")
for item, ok in checks.items():
    status = "✓" if ok else "✗ MISSING"
    print(f"  [{status}]  {item}")

AttributeError: 'CollectionInfo' object has no attribute 'vectors_count'